# 06 Housing supply and affordability

Reports what the HNR actually states about housing need, rental vacancy, ownership affordability, and short-term rental context. Does not attempt a rent-versus-resort-wage comparison: no source found in this project gives a resort-specific wage figure or a dollar-value renter affordability table (only an owner affordability table exists in the HNR), so that half of the step's Question is left as an open gap rather than approximated from unrelated figures.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"

## Load the ledger and step 02's housing-need table

Reads the component table step 02 already wrote instead of retyping its totals a third time.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C048"]["source_id"] == "S013"

housing_need = pd.read_csv(f"{processed_dir}/02_housing_need_by_component.csv")
housing_need

## Rental vacancy (claim C029)

One figure, not a table, so kept as plain values rather than a single-row DataFrame.

In [ ]:
rental_vacancy = {
    "vacancy_rate_2021_pct": 1.4,
    "healthy_vacancy_low_pct": 3,
    "healthy_vacancy_high_pct": 5,
    "renter_households_2021": 845,
    "units_needed_for_3pct_vacancy": 14,
}
rental_vacancy

## Ownership affordability (claim C048)

The HNR's own affordability figure gives a shelter-affordability gap by family type and dwelling type; kept as a full table since the gap sign (negative = affordable, positive = unaffordable at the report's stated income threshold) differs by family and dwelling type, not just as a headline sale-price statistic.

In [ ]:
ownership_affordability_gap = pd.DataFrame(
    [
        {"family_type": "Couples without children", "median_income": 116298,
         "single_detached_gap": -2200, "townhouse_gap": -1777, "condo_gap": -307},
        {"family_type": "Couples with children", "median_income": 153889,
         "single_detached_gap": -1260, "townhouse_gap": -837, "condo_gap": 632},
        {"family_type": "Lone parent families", "median_income": 81056,
         "single_detached_gap": -3081, "townhouse_gap": -2658, "condo_gap": -1188},
        {"family_type": "Non-census families", "median_income": 64845,
         "single_detached_gap": -3487, "townhouse_gap": -3063, "condo_gap": -1594},
        {"family_type": "Other census families", "median_income": 186782,
         "single_detached_gap": -438, "townhouse_gap": -14, "condo_gap": 1455},
    ]
)
# Negative gap = monthly shelter cost exceeds the affordable threshold (30% of income).
ownership_affordability_gap["can_afford_any_type"] = (
    ownership_affordability_gap[["single_detached_gap", "townhouse_gap", "condo_gap"]] > 0
).any(axis=1)
ownership_affordability_gap

## Short-term rental context (claims C013, C037, C014)

Recorded as plain facts, not a judgment about whether the exemption should change: RMR and the municipality of Revelstoke are both named exempt from the provincial principal-residence requirement, while the surrounding rural electoral area is not.

In [ ]:
str_context = {
    "municipality_exempt": True,
    "rmr_resort_area_exempt": True,
    "surrounding_rural_electoral_area_exempt": False,
}
str_context

## Write outputs

One file per topic, mirroring notebook 02's pattern.

In [ ]:
import json
import os

os.makedirs(processed_dir, exist_ok=True)
ownership_affordability_gap.to_csv(
    f"{processed_dir}/06_ownership_affordability_gap.csv", index=False, encoding="utf-8"
)
with open(f"{processed_dir}/06_rental_vacancy.json", "w", encoding="utf-8") as f:
    json.dump(rental_vacancy, f, indent=2)
with open(f"{processed_dir}/06_str_context.json", "w", encoding="utf-8") as f:
    json.dump(str_context, f, indent=2)
print("wrote 06_ownership_affordability_gap.csv, 06_rental_vacancy.json, 06_str_context.json")

## Checks

The healthy vacancy range's low bound must be under its high bound, the current vacancy rate must be below that range (that is the whole reason units are needed), and at least one row of the affordability table should show a family type that can afford at least one dwelling type, since otherwise the table would be uniformly unaffordable and worth double-checking.

In [ ]:
assert rental_vacancy["healthy_vacancy_low_pct"] < rental_vacancy["healthy_vacancy_high_pct"]
assert rental_vacancy["vacancy_rate_2021_pct"] < rental_vacancy["healthy_vacancy_low_pct"]
assert ownership_affordability_gap["can_afford_any_type"].any(), (
    "expected at least one family type to afford at least one dwelling type"
)
print("checks passed")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))